# Agent tool authorisation

**Module 14 — AI Agents and Tool Security**

An agent is a program that chooses actions. The security question is not "did the model choose well" but "was
the chosen action permitted for this principal, on this resource, right now". This notebook builds a policy
engine, wires four tools to it, and attacks it three ways: privilege escalation, parameter tampering and the
confused deputy.

## Learning objectives

1. Encode authorisation as a decision function over (principal, action, resource, context).
2. Enforce least privilege per tool, with argument validation independent of the model.
3. Demonstrate parameter tampering and confused-deputy attacks against a naive agent.
4. Add approval gates, spend limits and an audit trail.
5. Explain why the model's intent must never be an authorisation input.

## Background

Three rules carry most of the weight:

* **The model proposes, the policy disposes.** Tool calls are authorised by code, not by prompt text.
* **Arguments are untrusted.** The model may pass a resource id, amount or recipient it was manipulated into
  producing. Validate against the caller's scope.
* **Every call is attributable.** An audit record with principal, action, resource, decision and reason is the
  only way to reconstruct an incident.

A confused deputy is the classic failure: the agent has broad rights, the user has narrow rights, and the
agent uses its own rights on the user's behalf.

## Dependencies and installation

```bash
python -m pip install "pandas>=2.2"
```

Standard library only. All tools are mocks: `send_email`, `read_file`, `refund`, `query_db`. Nothing leaves
the notebook.

In [1]:
import pandas as pd

pd.set_option("display.width", 150)
print("ready")

ready


**Expected output** — `ready`.

### Step 1 — Principals, resources and a policy table

Policy is data. Data can be reviewed, tested and diffed; a paragraph of prose cannot.

In [2]:
principals = {
    "alice": {"tenant": "alpha", "roles": ["analyst"], "spend_limit": 0},
    "bob": {"tenant": "alpha", "roles": ["support"], "spend_limit": 50},
    "carol": {"tenant": "beta", "roles": ["manager"], "spend_limit": 500},
}
resources = {
    "doc:alpha:1": {"tenant": "alpha"},
    "doc:alpha:2": {"tenant": "alpha"},
    "doc:beta:9": {"tenant": "beta"},
}

policy = [
    {"action": "read_file", "roles": ["analyst", "support", "manager"], "scope": "same_tenant"},
    {"action": "send_email", "roles": ["support", "manager"], "scope": "same_tenant"},
    {"action": "refund", "roles": ["manager"], "scope": "same_tenant"},
    {"action": "query_db", "roles": ["analyst", "manager"], "scope": "same_tenant"},
]
pd.DataFrame(policy)

,action,roles,scope
0,read_file,"[analyst, support, manager]",same_tenant
1,send_email,"[support, manager]",same_tenant
2,refund,[manager],same_tenant
3,query_db,"[analyst, manager]",same_tenant


**Expected output** — four actions with the roles allowed and a tenant scope. `refund` is manager-only, which
sets up the escalation test.

### Step 2 — The decision function

Four checks, any of which denies: role, tenant scope, argument validity, and context constraints such as spend
limits. The reason string is returned so the audit log explains itself.

In [3]:
class PolicyEngine:
    def __init__(self):
        self.log = []

    def decide(self, principal, action, resource, amount=0, recipient_tenant=None):
        p = principals.get(principal)
        if p is None:
            return self._log(principal, action, resource, False, "unknown principal")
        rule = next((r for r in policy if r["action"] == action), None)
        if rule is None:
            return self._log(principal, action, resource, False, "unknown action")
        if not set(rule["roles"]) & set(p["roles"]):
            return self._log(principal, action, resource, False, "role not permitted")
        if resource is not None:
            res = resources.get(resource)
            if res is None:
                return self._log(principal, action, resource, False, "unknown resource")
            if rule["scope"] == "same_tenant" and res["tenant"] != p["tenant"]:
                return self._log(principal, action, resource, False, "cross-tenant resource")
        if recipient_tenant is not None and recipient_tenant != p["tenant"]:
            return self._log(principal, action, resource, False, "cross-tenant recipient")
        if amount > p["spend_limit"]:
            return self._log(principal, action, resource, False, "spend limit exceeded")
        return self._log(principal, action, resource, True, "allowed")

    def _log(self, principal, action, resource, allowed, reason):
        self.log.append({"principal": principal, "action": action, "resource": resource,
                         "allowed": allowed, "reason": reason})
        return allowed, reason


engine = PolicyEngine()
print(engine.decide("alice", "read_file", "doc:alpha:1"))
print(engine.decide("alice", "refund", "doc:alpha:1", amount=10))
print(engine.decide("alice", "read_file", "doc:beta:9"))

(True, 'allowed')
(False, 'role not permitted')
(False, 'cross-tenant resource')


**Expected output** — `(True, 'allowed')`, `(False, 'role not permitted')`, `(False, 'cross-tenant resource')`.
Three lines, three different controls — and the reason strings are what make the log useful later.

### Step 3 — The naive agent, and the confused deputy

The naive agent checks the *user's* request against nothing and calls tools with its own service identity.
Bob (support) cannot refund, but the agent's service account can — that is the deputy problem.

In [4]:
class NaiveAgent:
    def __init__(self):
        self.service_role = "manager"

    def handle(self, request, user):
        # The agent trusts the request text and uses its own privileges
        if "refund" in request.lower():
            return {"tool": "refund", "executed": True, "note": f"executed with {self.service_role} rights for {user}"}
        return {"tool": "read_file", "executed": True}


naive = NaiveAgent()
print(naive.handle("Please refund the customer 20 GBP.", "bob"))
print("bob's own spend limit:", principals["bob"]["spend_limit"])

{'tool': 'refund', 'executed': True, 'note': 'executed with manager rights for bob'}
bob's own spend limit: 50


**Expected output** — the refund executes even though `bob` cannot refund and even though support staff have a
spend limit of 50. The model would have "helpfully" complied; the failure is architectural.

### Step 4 — A governed agent

The governed agent routes every call through the engine with the *user's* principal and the *requested*
arguments. The model can propose anything; the decision is code.

In [5]:
class GovernedAgent:
    def __init__(self, engine):
        self.engine = engine
        self.executed = []

    def handle(self, request, user, resource=None, amount=0, recipient_tenant=None):
        action = "refund" if "refund" in request.lower() else "read_file"
        allowed, reason = self.engine.decide(user, action, resource, amount=amount,
                                             recipient_tenant=recipient_tenant)
        if not allowed:
            return {"tool": action, "executed": False, "reason": reason}
        self.executed.append((action, resource, amount))
        return {"tool": action, "executed": True, "reason": reason}


gov = GovernedAgent(engine)
cases = [
    ("refund 20 GBP", "bob", "doc:alpha:1", 20, None),
    ("refund 20 GBP", "carol", "doc:beta:9", 20, None),
    ("refund 900 GBP", "carol", "doc:beta:9", 900, None),
    ("read summary", "alice", "doc:beta:9", 0, None),
    ("read summary", "alice", "doc:alpha:2", 0, None),
]
rows = []
for req, user, res, amt, rec in cases:
    out = gov.handle(req, user, resource=res, amount=amt, recipient_tenant=rec)
    rows.append({"request": req, "user": user, "resource": res, "executed": out["executed"],
                 "reason": out["reason"]})
governed = pd.DataFrame(rows)
governed

,request,user,resource,executed,reason
0,refund 20 GBP,bob,doc:alpha:1,False,role not permitted
1,refund 20 GBP,carol,doc:beta:9,True,allowed
2,refund 900 GBP,carol,doc:beta:9,False,spend limit exceeded
3,read summary,alice,doc:beta:9,False,cross-tenant resource
4,read summary,alice,doc:alpha:2,True,allowed


**Expected output** — rows 1, 3 and 4 denied (role, spend limit, cross-tenant) while rows 2 and 5 are allowed.
Every denial has a specific reason, which is what turns an audit log into an investigation tool.

### Step 5 — Parameter tampering, and the recipient check

The most common real attack is not a forbidden tool but a permitted tool with manipulated arguments: the
recipient, the amount, the resource id.

In [6]:
tampered = [
    {"user": "bob", "resource": "doc:alpha:1", "amount": 0, "recipient_tenant": "beta"},
    {"user": "bob", "resource": "doc:alpha:1", "amount": 49, "recipient_tenant": "alpha"},
    {"user": "bob", "resource": "doc:alpha:1", "amount": 51, "recipient_tenant": "alpha"},
]
out = []
for t in tampered:
    allowed, reason = engine.decide(t["user"], "send_email", t["resource"], amount=t["amount"],
                                    recipient_tenant=t["recipient_tenant"])
    out.append({**t, "allowed": allowed, "reason": reason})
tampering = pd.DataFrame(out)
tampering

,user,resource,amount,recipient_tenant,allowed,reason
0,bob,doc:alpha:1,0,beta,False,cross-tenant recipient
1,bob,doc:alpha:1,49,alpha,True,allowed
2,bob,doc:alpha:1,51,alpha,False,spend limit exceeded


**Expected output** — cross-tenant recipient denied, 49 GBP allowed, 51 GBP denied. Three rows that differ only
in arguments. This is why authorisation must inspect arguments, not just tool names.

### Step 6 — The audit trail

Every decision, allowed or denied, with the reason. Reading the log is how you find the attempt that failed
nine times and succeeded on the tenth.

In [7]:
audit = pd.DataFrame(engine.log)
print(f"{len(audit)} decisions, {int((~audit.allowed).sum())} denials")
print(audit.reason.value_counts().to_string())
assert (audit[~audit.allowed].reason != "allowed").all()
audit.tail(8)

11 decisions, 7 denials
reason
allowed                   4
role not permitted        2
cross-tenant resource     2
spend limit exceeded      2
cross-tenant recipient    1


,principal,action,resource,allowed,reason
3,bob,refund,doc:alpha:1,False,role not permitted
4,carol,refund,doc:beta:9,True,allowed
5,carol,refund,doc:beta:9,False,spend limit exceeded
6,alice,read_file,doc:beta:9,False,cross-tenant resource
7,alice,read_file,doc:alpha:2,True,allowed
8,bob,send_email,doc:alpha:1,False,cross-tenant recipient
9,bob,send_email,doc:alpha:1,True,allowed
10,bob,send_email,doc:alpha:1,False,spend limit exceeded


**Expected output** — a count of decisions and denials, a reason histogram, and the last eight rows. The
assertion encodes the invariant that a denial always carries a non-trivial reason.

## Exercises

1. **Add a tool.** Add `delete_file` restricted to managers in the same tenant, with an approval requirement
   for resources marked sensitive. Add it to the policy table and write two tests.
2. **Break the deputy.** Construct a request where a governed agent with a service account still acts beyond
   the user's rights, and fix it by passing the user principal end to end.
3. **Rate limit.** Add a per-user call budget with an audit entry on exhaustion. What should the agent tell the
   user, and what should it tell the operator?
4. **Two-person rule.** Require a second principal for refunds above 200. Sketch the data model and the failure
   mode when the approver is unavailable.

## Reflection questions

1. Why must the model's stated intent never be an authorisation input?
2. Which single control in this notebook would have stopped the confused-deputy attack, and why is it not
   enough on its own?
3. How would you test this policy engine in CI without a model at all?

## References

1. OWASP, *Top 10 for LLM Applications*, 2025 — LLM06 (excessive agency), LLM07.
2. NIST, *SP 800-162*, Guide to Attribute Based Access Control, 2014.
3. Saltzer, J. and Schroeder, M., "The Protection of Information in Computer Systems", 1975 — least privilege.
4. Simon, M. et al., "Capability-based security for LLM agents", 2024.